# 10 - Vida Util STJ Enriquecida

Objetivo: juntar ATA do STJ, movimentos do DataJud STJ e documentos do corpus do STJ para construir uma timeline unificada da vida util processual no STJ.

## 1. Ambiente e entradas

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    MOUNT_POINT = Path('/content/drive')
    if not (MOUNT_POINT / 'MyDrive').exists():
        drive.mount(str(MOUNT_POINT), force_remount=True)
    DATA_ROOT = MOUNT_POINT / 'MyDrive/Mestrado/2026/llms/data'
else:
    DATA_ROOT = Path.cwd().resolve().parents[0] / 'data'

PROCESSED_DATA = DATA_ROOT / 'processed'

process_spine = pd.read_parquet(PROCESSED_DATA / 'stj_processos_ciclo_vida.parquet')
ata_partes = pd.read_parquet(PROCESSED_DATA / 'stj_ata_partes.parquet')
ata_advogados = pd.read_parquet(PROCESSED_DATA / 'stj_ata_advogados.parquet')
documentos = pd.read_parquet(PROCESSED_DATA / 'stj_documentos_por_processo.parquet')
datajud_processos = pd.read_parquet(PROCESSED_DATA / 'stj_datajud_processos.parquet')
datajud_movimentos = pd.read_parquet(PROCESSED_DATA / 'stj_datajud_movimentos.parquet')

print('process_spine:', process_spine.shape)
print('documentos:', documentos.shape)
print('datajud_movimentos:', datajud_movimentos.shape)

process_spine: (2582, 19)
documentos: (84224, 38)
datajud_movimentos: (613, 9)


In [2]:
def parse_mixed_datetime(series):
    as_str = series.astype("string").str.strip()

    # primeiro tenta o parser geral em UTC
    dt = pd.to_datetime(as_str, errors="coerce", utc=True)

    # fallback para formato compacto YYYYMMDDHHMMSS
    missing = dt.isna() & as_str.notna()
    if missing.any():
        dt.loc[missing] = pd.to_datetime(
            as_str.loc[missing],
            format="%Y%m%d%H%M%S",
            errors="coerce",
            utc=True,
        )

    # remove timezone para tudo ficar homogêneo
    return dt.dt.tz_localize(None)


## 2. Normalizar eventos

In [3]:
ata_events = process_spine[['numero_processo', 'numero_registro_stj', 'data_primeira_distribuicao_stj', 'formas_distribuicao_stj', 'destinos_stj']].copy()
ata_events["event_datetime"] = parse_mixed_datetime(ata_events["data_primeira_distribuicao_stj"])
ata_events['event_source'] = 'ata_stj'
ata_events['event_type'] = 'distribuicao'
ata_events['event_label'] = ata_events['formas_distribuicao_stj']
ata_events['event_detail'] = ata_events['destinos_stj']

mov_events = datajud_movimentos[['numero_processo', 'numero_registro_stj', 'data_movimento', 'codigo_movimento', 'nome_movimento', 'complementos_tabelados']].copy()
mov_events["event_datetime"] = parse_mixed_datetime(mov_events["data_movimento"])
mov_events['event_source'] = 'datajud_stj'
mov_events['event_type'] = 'movimento'
mov_events['event_label'] = mov_events['nome_movimento']
mov_events['event_detail'] = mov_events['complementos_tabelados']

doc_events = documentos[['numero_processo', 'numero_registro_stj', 'data_documento', 'seq_documento', 'tipo_documento', 'ministro_documento']].copy()
doc_events["event_datetime"] = parse_mixed_datetime(doc_events["data_documento"])
doc_events['event_source'] = 'integra_stj'
doc_events['event_type'] = 'documento'
doc_events['event_label'] = doc_events['tipo_documento']
doc_events['event_detail'] = doc_events['ministro_documento']


## 3. Unificar timeline

In [4]:
common_cols = ['numero_processo', 'numero_registro_stj', 'event_datetime', 'event_source', 'event_type', 'event_label', 'event_detail']
ata_events = ata_events[common_cols]
mov_events = mov_events[common_cols]
doc_events = doc_events[common_cols + ['seq_documento']]

process_events = pd.concat([ata_events, mov_events, doc_events], ignore_index=True, sort=False)
process_events['processo_agregacao'] = process_events['numero_processo'].combine_first(process_events['numero_registro_stj'])
process_events = process_events.sort_values(['processo_agregacao', 'event_datetime', 'event_source']).reset_index(drop=True)
process_events.head(20)

,numero_processo,numero_registro_stj,event_datetime,event_source,event_type,event_label,event_detail,seq_documento,processo_agregacao
0,00000094820228272722,202301327446,2023-04-24 11:54:32,datajud_stj,movimento,Recebimento,None,NaN,00000094820228272722
1,00000094820228272722,202301327446,2023-04-27 16:15:08,datajud_stj,movimento,Distribuição,"[{""codigo"": 2, ""valor"": 1, ""nome"": ""competênci...",NaN,00000094820228272722
2,00000094820228272722,202301327446,2023-04-27 16:30:05,datajud_stj,movimento,Conclusão,"[{""codigo"": 3, ""valor"": 6, ""nome"": ""para decis...",NaN,00000094820228272722
3,00000094820228272722,202301327446,2023-05-15 18:30:17,datajud_stj,movimento,Mero expediente,None,NaN,00000094820228272722
4,00000094820228272722,202301327446,2023-05-15 18:30:17,datajud_stj,movimento,Ato ordinatório,None,NaN,00000094820228272722
5,00000094820228272722,202301327446,2023-05-16 19:11:45,datajud_stj,movimento,Disponibilização no Diário da Justiça Eletrônico,None,NaN,00000094820228272722
6,00000094820228272722,202301327446,2023-05-17 05:29:08,datajud_stj,movimento,Publicação,None,NaN,00000094820228272722
7,00000094820228272722,202301327446,2023-05-23 19:28:06,datajud_stj,movimento,Protocolo de Petição,None,NaN,00000094820228272722
8,00000094820228272722,202301327446,2023-05-23 19:28:12,datajud_stj,movimento,Recebimento,None,NaN,00000094820228272722
9,00000094820228272722,202301327446,2023-05-23 19:31:00,datajud_stj,movimento,Petição,"[{""codigo"": 19, ""valor"": 122, ""nome"": ""Parecer...",NaN,00000094820228272722


## 4. Agregar por processo

In [5]:
timeline_rows = []
for processo_key, group in process_events.groupby('processo_agregacao', dropna=True):
    records = group[['event_datetime', 'event_source', 'event_type', 'event_label', 'event_detail']].copy()
    records['event_datetime'] = records['event_datetime'].astype(str)
    timeline_rows.append({
        'processo_agregacao': processo_key,
        'numero_processo': group['numero_processo'].dropna().astype(str).iloc[0] if group['numero_processo'].notna().any() else None,
        'numero_registro_stj': group['numero_registro_stj'].dropna().astype(str).iloc[0] if group['numero_registro_stj'].notna().any() else None,
        'timeline_event_count': len(group),
        'timeline_start': group['event_datetime'].min(),
        'timeline_end': group['event_datetime'].max(),
        'timeline_json': json.dumps(records.to_dict('records'), ensure_ascii=False),
    })

process_timeline = pd.DataFrame(timeline_rows)
process_timeline.head()

,processo_agregacao,numero_processo,numero_registro_stj,timeline_event_count,timeline_start,timeline_end,timeline_json
0,00000094820228272722,00000094820228272722,202301327446,120,2023-04-24 11:54:32,2024-11-21 14:23:04,"[{""event_datetime"": ""2023-04-24 11:54:32"", ""ev..."
1,00000195520128020001,00000195520128020001,202302129700,58,2023-06-21 09:15:40,2024-02-26 18:53:06,"[{""event_datetime"": ""2023-06-21 09:15:40"", ""ev..."
2,00000270520134047105,00000270520134047105,201700539640,67,2017-03-15 19:24:06,2023-10-27 18:23:06,"[{""event_datetime"": ""2017-03-15 19:24:06"", ""ev..."
3,00000461120218120012,00000461120218120012,202302183487,60,2023-06-23 16:42:03,2025-12-15 18:51:07,"[{""event_datetime"": ""2023-06-23 16:42:03"", ""ev..."
4,00000675420208190035,00000675420208190035,202300281603,24,2023-02-02 13:21:43,2023-09-19 14:01:40,"[{""event_datetime"": ""2023-02-02 13:21:43"", ""ev..."


## 5. Salvar artefatos

In [6]:
process_events.to_parquet(PROCESSED_DATA / 'stj_process_events.parquet', index=False)
process_timeline.to_parquet(PROCESSED_DATA / 'stj_process_timeline.parquet', index=False)

print('Salvos:')
print(PROCESSED_DATA / 'stj_process_events.parquet')
print(PROCESSED_DATA / 'stj_process_timeline.parquet')

Salvos:
/content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_process_events.parquet
/content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_process_timeline.parquet
